In [ ]:
from pathlib import Path
import glob
import os
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import MultipleLocator
import pandas as pd

from thomas2024.calibration import assemble_thresholds

In [ ]:
# Input data paths
country_thresholds_path = Path("../data/20260907_country_thresholds.csv")
income_thresholds_path = Path("../data/20260907_income_thresholds.csv")
wb_income_path = Path("../data/OGHIST_2026_07_15.xlsx")
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = Path("../plots")
plot_dir.mkdir(parents=True, exist_ok=True)

# Do not extrapolate beyond the available events
tc_model_n_years = {
    "CHAZ": 1000,
    "MIT": 200,
    "IRIS": 1000,
    "STORM": 1000,
}

# Mapping from open-gira STORM_SET to display name
atmospheres_to_scenario = {
    "STORM_constant": "2000 STORM",
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3",
    "IRIS_PRESENT": "2000 IRIS",
    "IRIS_SSP5-2050": "2050 IRIS SSP5-8.5",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": "2010 CHAZ UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": "2010 CHAZ CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": "2010 CHAZ CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": "2010 CHAZ EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": "2010 CHAZ MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2005": "2005 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": "2005 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": "2005 Emanuel EC-Earth6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": "2005 Emanuel MIROC6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": "2005 Emanuel UKMO6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "IRIS baseline": ["2000 IRIS"],
    "IRIS SSP5-8.5 2050": ["2050 IRIS SSP5-8.5"],
    "STORM baseline": ["2000 STORM"],
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM",
        "2050 STORM RCP8.5 CMCC",
        "2050 STORM RCP8.5 EC-Earth",
        "2050 STORM RCP8.5 HadGEM3"
    ],
    "CHAZ baseline": [
        "2010 CHAZ UKESM1",
        "2010 CHAZ CESM2",
        "2010 CHAZ CNRM",
        "2010 CHAZ EC-Earth3",
        "2010 CHAZ MIROC6",
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 UKESM1",
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
    ],   
    "MIT baseline": [
        "2005 Emanuel CESM2",
        "2005 Emanuel CNRM6",
        "2005 Emanuel EC-Earth6",
        "2005 Emanuel MIROC6",
        "2005 Emanuel UKMO6",
    ],
    "MIT SSP5-8.5 2050": [
        "2050 Emanuel CESM2",
        "2050 Emanuel CNRM6",
        "2050 Emanuel EC-Earth6",
        "2050 Emanuel MIROC6",
        "2050 Emanuel UKMO6",
    ],
}
group_colours = {
    "IRIS": "lightgreen",
    "STORM": "plum",
    "CHAZ": "pink",
    "MIT": "lightblue",
}

# Assemble country-specific calibrated thresholds from income and country data.
thresholds_by_country = assemble_thresholds(
    wb_income_path, income_thresholds_path, country_thresholds_path
)

#rps = np.array([i * base for base in (1, 10, 100) for i in range(1, 10)])
rps = np.logspace(0, 3, 40)

In [ ]:
def compute_country_pop_affected_by_rp(
    atmospheres_to_scenario,
    gcm_groups,
    tc_model_n_years,
    thresholds_by_country,
    results_dir,
    rps,
):
    """
    Build country-level pop_affected curves interpolated onto a common
    return_period_years grid, keeping TC model / Scenario / Country
    information intact.

    Returns
    -------
    baseline, future : pd.DataFrame
        Each indexed by MultiIndex (TC model, Scenario, Country,
        return_period_years) with a single column 'pop_affected'.
    """
    baseline_frames = []
    future_frames = []

    for atmosphere, scenario in atmospheres_to_scenario.items():

        print(atmosphere, end="\r")
        paths = glob.glob(os.path.join(
            results_dir, f"power/by_country/*/disruption/{atmosphere}/pop_affected_by_event.pq"
        ))
        data = {path.split("/")[-4]: pd.read_parquet(path) for path in paths}

        tc_model = atmosphere.split("_")[0]
        if tc_model == "emanuel":
            tc_model = "MIT"
        years_of_events = tc_model_n_years[tc_model]

        # Work out which epoch (baseline/future) this scenario belongs to
        for group_name, members in gcm_groups.items():
            if scenario in members:
                break
        family, *other = group_name.split()
        epoch = "baseline" if other[0] == "baseline" else "future"

        for iso_a3, df in data.items():
            try:
                threshold = thresholds_by_country.loc[iso_a3, "threshold_ms-1"]
                series = df.loc[:, str(threshold)]
            except KeyError:
                continue

            cdf = series.rename("pop_affected").reset_index()

            if atmosphere.startswith("STORM") or atmosphere.startswith("IRIS"):
                cdf["year"] = cdf.event_id.apply(lambda s: int(s.split("_")[2]))
            elif atmosphere.startswith("CHAZ") or atmosphere.startswith("emanuel"):
                cdf["year"] = cdf.event_id.apply(lambda s: int(re.search(r"Y(\d{4})", s).groups()[0]))

            annual_sum = cdf.drop(columns=["event_id"]).groupby("year")["pop_affected"].sum()

            # Use the known simulation length rather than inferring it from
            # this country's data -- a country can have zero affected
            # population (and therefore no rows at all) in many years.
            n_years = years_of_events
            full_years = pd.RangeIndex(0, n_years, name="year")
            annual_sum = annual_sum.reindex(full_years, fill_value=0.0)

            cdf = annual_sum.reset_index().sort_values("pop_affected").reset_index(drop=True)

            year_range = range(n_years - 1, -1, -1)
            cdf["rank"] = year_range
            cdf["return_period_years"] = (n_years + 1) / cdf["rank"]

            cdf = cdf.drop(columns=["year", "rank"])
            cdf = cdf[np.isfinite(cdf.return_period_years)]
            cdf["log_rp"] = np.log(cdf.return_period_years)

            target = pd.DataFrame({
                "return_period_years": rps,
                "log_rp": np.log(rps),
                "pop_affected": np.nan,
            })
            cdf = pd.concat([cdf, target]).sort_values("log_rp").set_index("log_rp")
            cdf = cdf.interpolate(method="index")
            cdf = cdf.loc[np.log(rps)].reset_index(drop=True)

            cdf.loc[cdf.return_period_years > years_of_events, "pop_affected"] = np.nan

            cdf["TC model"] = tc_model
            cdf["Scenario"] = scenario
            cdf["Country"] = iso_a3

            cdf = cdf.set_index(["TC model", "Scenario", "Country", "return_period_years"])

            if epoch == "baseline":
                baseline_frames.append(cdf)
            else:
                future_frames.append(cdf)

    baseline = pd.concat(baseline_frames) if baseline_frames else pd.DataFrame()
    future = pd.concat(future_frames) if future_frames else pd.DataFrame()

    return baseline, future


def collapse_scenario_mean(df):
    """
    Collapse the 'Scenario' level of the MultiIndex by averaging, leaving
    one value per (TC model, Country, return_period_years).
    """
    return df.groupby(level=["TC model", "Country", "return_period_years"]).mean()

baseline, future = compute_country_pop_affected_by_rp(
    atmospheres_to_scenario, gcm_groups, tc_model_n_years, thresholds_by_country, results_dir, rps
)

baseline_mean = collapse_scenario_mean(baseline)
future_mean = collapse_scenario_mean(future)

df = future_mean.rename(columns={"pop_affected": "future"}).join(baseline_mean.rename(columns={"pop_affected": "baseline"}))
df["ratio"] = df.future / df.baseline
df["diff"] = df.future - df.baseline

In [ ]:
# Top N largest changes (by magnitude) in risk by TC model and country

f, axes = plt.subplots(2, 2, figsize=(8, 5.5))
top_n = 12
rp = 50
tc_models = df.index.get_level_values("TC model").unique()

increase_hatch = ""      # solid fill for increases
decrease_hatch = "///"   # hatched fill for decreases

# Build candidate country set from the top-n-by-magnitude changes, across all models
candidate_countries = set()
for tc_model in tc_models:
    at_rp = (
        df.loc[tc_model, :, rp - 1: rp + 1, :]
            .droplevel("TC model")
            .droplevel("return_period_years")["diff"]
    )
    top_change = at_rp.reindex(at_rp.abs().sort_values(ascending=False).index).iloc[:top_n]
    candidate_countries.update(top_change.index)

candidate_countries = sorted(candidate_countries)

cmap = plt.get_cmap("tab20c", len(candidate_countries))
country_colors = {
    country: cmap(i) for i, country in enumerate(candidate_countries)
}

global_max = df.loc[:, :, rp - 1: rp + 1]["diff"].max()
global_min = df.loc[:, :, rp - 1: rp + 1]["diff"].min()
x_buffer = 0.1 * global_max

for i, (tc_model, ax) in enumerate(zip(tc_models, axes.ravel())):
    at_rp = (
        df.loc[tc_model, :, rp - 1: rp + 1, :]
            .droplevel("TC model")
            .droplevel("return_period_years")["diff"]
    )

    # Select top_n by absolute value, then sort by signed value for plotting
    top_change = at_rp.reindex(at_rp.abs().sort_values(ascending=False).index).iloc[:top_n]
    top_change = top_change.sort_values()

    colors = [country_colors[c] for c in top_change.index]
    hatches = [decrease_hatch if v < 0 else increase_hatch for v in top_change]

    bars = ax.barh(top_change.index, top_change.values, color=colors)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
        bar.set_edgecolor("black")
        bar.set_linewidth(0.5)

    ax.set_yticks([])
    ax.bar_label(bars, top_change.index, padding=4, size=7)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.text(0.05, 0.88, tc_model, transform=ax.transAxes, size=10)
    ax.set_ylabel("")
    ax.set_xlim(global_min - x_buffer, global_max + x_buffer)
    #ax.set_xlim(global_min - x_buffer, global_max)
    ax.grid(axis="x", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    if i < 2:
        ax.spines['bottom'].set_visible(False)
        ax.set_xticklabels([])
    else:
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e7:.0f}"))

f.supxlabel(r"Largest future changes in pop. at risk ($\times 10^{7}$)", y=0.1, size=10)
plt.subplots_adjust(left=0.07, bottom=0.22, right=0.95, hspace=0.15, wspace=0.1)
f.savefig(plot_dir / f"pop_at_risk_largest_changes_1-in-{rp}_by_model.pdf", bbox_inches="tight")

In [ ]:
top_n = 13
rp = 50
drop_countries = ["NIC"]
tc_models = df.index.get_level_values("TC model").unique()

baseline = {}
for tc_model in tc_models:
    baseline[tc_model] = (
        df.loc[tc_model, :, rp - 1:rp + 1, :]
        .droplevel(["TC model", "return_period_years"])["baseline"]
    )
baseline = pd.DataFrame(baseline).drop(index=drop_countries, errors="ignore")
countries = baseline.abs().median(axis=1).nlargest(top_n).index
baseline = baseline.loc[countries]

changes = {}
for tc_model in tc_models:
    changes[tc_model] = (
        df.loc[tc_model, :, rp - 1:rp + 1, :]
        .droplevel(["TC model", "return_period_years"])["diff"]
    )
changes = pd.DataFrame(changes).drop(index=drop_countries, errors="ignore")
changes = changes.loc[baseline.index]

ranges = [changes.max().max() - changes.min().min(), baseline.max().max() - 0]
f, (axc, ax) = plt.subplots(2, 1, figsize=(7, 3.5), height_ratios=ranges)

x = np.arange(len(baseline))
bar_width = 0.7 / len(tc_models)
for i, tc_model in enumerate(tc_models):
    offset = (i - (len(tc_models) - 1) / 2) * bar_width
    ax.bar(
        x + offset,
        baseline[tc_model],
        width=bar_width,
        label=tc_model,
        color=group_colours[tc_model],
        edgecolor="black",
        linewidth=0.4
    )
    
for i, tc_model in enumerate(tc_models):
    offset = (i - (len(tc_models) - 1) / 2) * bar_width
    axc.bar(
        x + offset,
        changes[tc_model],
        width=bar_width,
        label=tc_model,
        color=group_colours[tc_model],
        edgecolor="black",
        linewidth=0.4
    )

for a in (ax, axc):
    a.spines[["top", "right", "bottom"]].set_visible(False)
    a.tick_params(axis="y", labelsize=7)
    a.tick_params(axis="y", which="minor", length=2.5)
    a.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y / 1e7:.1f}"))
    a.yaxis.set_minor_locator(MultipleLocator(1e6))
    a.yaxis.set_major_locator(MultipleLocator(1e7))
    a.tick_params(axis="x", length=0, labelsize=7)
    a.grid(axis="y", alpha=0.2, linewidth=0.5, which="both")
    a.set_axisbelow(True)
    a.axhline(0, color="black", linewidth=0.8)
    tick_x = x[:-1] + 0.5
    tick_len = 0.02 * (ax.get_ylim()[1] - ax.get_ylim()[0])
    a.vlines(tick_x, ymin=-tick_len, ymax=0, color="black", linewidth=0.8, clip_on=False)

ax.set_xticks(x, baseline.index, rotation=90, ha="center", fontsize=7)
axc.set_xticklabels([])
ax.set_ylabel(f"Baseline pop. at risk \n" + r"($\times 10^{7}$)", fontsize=9)
axc.set_ylabel(rf"Future $\Delta$ " + "\n" + r"($\times 10^{7}$)", fontsize=9)
ax.legend(frameon=False, fontsize=8, loc="upper right", handlelength=1.2)

f.subplots_adjust(left=0.18, right=0.995, bottom=0.23, top=1, hspace=0.05)
f.savefig(plot_dir / f"pop_at_risk_largest_changes_1-in-{rp}_grouped_rel_abs.pdf", bbox_inches="tight")

In [ ]:
baseline = {}
for tc_model in tc_models:
    baseline[tc_model] = (
        df.loc[tc_model, :, rp - 1:rp + 1, :]
        .droplevel(["TC model", "return_period_years"])["baseline"]
    )
baseline = pd.DataFrame(baseline).drop(index=drop_countries, errors="ignore")
x = baseline.median(axis=1).sort_values()
c = x / x.sum()
s = c.cumsum()
x = pd.DataFrame({"c": c, "s": s})
print(x.loc[countries, "c"].sum())
x.iloc[-16:]

In [ ]:
def risk_decomposition(g, col="baseline", edges=(1, 10, 50, 100)):
    g = g.sort_values("return_period_years")
    rp = g["return_period_years"].to_numpy()
    imp = g[col].fillna(0).to_numpy()
    f = 1.0 / rp

    seg = 0.5 * (imp[1:] + imp[:-1]) * (f[:-1] - f[1:])
    total = seg.sum()
    if total == 0:
        return pd.Series(dtype=float)

    cum = np.concatenate([[0], np.cumsum(seg)])
    cum_at = np.interp(np.log(edges), np.log(rp), cum)  # interpolate in log-RP

    out = {"EAPA": total}
    bounds = list(edges)
    for lo, hi, c_lo, c_hi in zip(bounds[:-1], bounds[1:], cum_at[:-1], cum_at[1:]):
        out[f"share_RP_{lo}-{hi}"] = (c_hi - c_lo) / total
    out[f"share_RP_>{bounds[-1]}"] = (total - cum_at[-1]) / total

    out["RP_50pct_of_EAPA"] = np.exp(np.interp(0.5 * total, cum, np.log(rp)))
    return pd.Series(out)

max_years = 1000
models = ("CHAZ", "IRIS", "STORM", "MIT")
drop_countries = ["NIC", "GTM", "MOZ", "MDG"]
sample = df[df.index.get_level_values("return_period_years") < max_years]
sample = sample[sample.index.get_level_values("TC model").isin(models)]
sample = sample[~sample.index.get_level_values("Country").isin(drop_countries)]
res = (sample.reset_index("return_period_years").groupby(level=["TC model", "Country"])
         .apply(risk_decomposition))
res.index = res.index.set_names("var", level=2)

comp_mean = (
    res
    .groupby(level=["Country", "var"])
    .mean()
    .unstack("var")
)
share_cols = [c for c in comp_mean.columns if c.startswith("share_RP_")]
comp_mean.sort_values("RP_50pct_of_EAPA").style.highlight_max(
    subset=share_cols,
    axis=1,
    color="darkred",
)

In [ ]:
# Saturation = the first sampled RP of a plateau that persists to the last
# finite baseline value, with at least two points on that plateau.
# This detects saturation only within the observed RP range, not beyond it.
saturation_rtol = 0.01
min_plateau_points = 2

# Drop country-model pairs with only zero/missing baseline values.
has_nonzero_baseline = (
    df["baseline"].fillna(0).ne(0)
    .groupby(level=["Country", "TC model"]).transform("any")
)
saturation_df = df.copy()
saturation_df = saturation_df.loc[has_nonzero_baseline]

# Exclude MIT because it has fewer available return periods.
# saturation_df = saturation_df.loc[(saturation_df.index.get_level_values("TC model") != "MIT")]

saturation_rows = []
for (country, tc_model), group in saturation_df.groupby(level=["Country", "TC model"]):
    curve = group["baseline"].droplevel(["Country", "TC model"]).sort_index()
    # Exclude missing values, including RPs beyond a model's simulation length.
    curve = curve[np.isfinite(curve)]
    plateau_start = len(curve)
    if len(curve):
        near_final = np.isclose(
            curve.to_numpy(), curve.iloc[-1],
            rtol=saturation_rtol,
        )
        # Only the final uninterrupted run counts; earlier flat sections do not.
        outside_plateau = np.flatnonzero(~near_final)
        plateau_start = outside_plateau[-1] + 1 if len(outside_plateau) else 0

    plateau_points = len(curve) - plateau_start
    saturated = plateau_points >= min_plateau_points
    saturation_rows.append({
        "Country": country,
        "TC model": tc_model,
        "Status": (
            "Insufficient data" if len(curve) < min_plateau_points
            else "Saturated" if saturated else "Not observed"
        ),
        "Saturation RP (years)": curve.index[plateau_start] if saturated else np.nan,
        "Plateau baseline": curve.iloc[-1] if saturated else np.nan,
        "Max observed RP (years)": curve.index[-1] if len(curve) else np.nan,
        "Plateau points": plateau_points if saturated else 0,
    })

saturation_results = (
    pd.DataFrame(saturation_rows, columns=[
        "Country", "TC model", "Status", "Saturation RP (years)",
        "Plateau baseline", "Max observed RP (years)", "Plateau points",
    ]).set_index(["Country", "TC model"]).sort_index()
)
saturation_results.style.format({
    "Saturation RP (years)": "{:.2f}",
    "Plateau baseline": "{:,.2f}",
    "Max observed RP (years)": "{:.2f}",
}, na_rep="—")

In [ ]:
# Plot the country-model pairs retained in the saturation table, in table order.
from matplotlib.lines import Line2D

plot_countries = saturation_results.index.get_level_values("Country").unique()
plot_models = saturation_results.index.get_level_values("TC model").unique()

if len(plot_countries) == 0:
    print("No country-model pairs remain to plot.")
else:
    ncols = min(3, len(plot_countries))
    nrows = int(np.ceil(len(plot_countries) / ncols))
    saturation_fig, saturation_axes = plt.subplots(
        nrows, ncols, figsize=(4 * ncols, 2.8 * nrows),
        sharex=True, squeeze=False, layout="constrained",
    )
    # Use the same full RP range in every panel; each country has its own y-scale.
    plot_rps = df.index.get_level_values("return_period_years").to_numpy()
    plot_rps = plot_rps[np.isfinite(plot_rps) & (plot_rps > 0)]
    rp_limits = (plot_rps.min(), plot_rps.max())

    for country, ax in zip(plot_countries, saturation_axes.flat):
        country_results = saturation_results.xs(country, level="Country")
        for tc_model, result in country_results.iterrows():
            curve = saturation_df.xs(
                (tc_model, country), level=["TC model", "Country"]
            )["baseline"].sort_index()
            colour = group_colours[tc_model]
            ax.plot(curve.index, curve, color=colour, linewidth=1.8)

            sat_rp = result["Saturation RP (years)"]
            if pd.notna(sat_rp):
                # Mark the actual curve value at the detected RP, not the final value.
                sat_impact = curve.loc[sat_rp]
                ax.scatter(
                    sat_rp, sat_impact, color=colour, edgecolor="black",
                    linewidth=0.6, s=40, zorder=4,
                )
                ax.axvline(sat_rp, color=colour, linestyle=":", alpha=0.7)

        ax.set_title(country, fontsize=10)
        ax.set_xscale("log")
        ax.set_xlim(*rp_limits)
        ax.set_ylim(bottom=0)
        ax.tick_params(axis="x", labelbottom=True)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        ax.grid(alpha=0.2, which="both")
        ax.spines[["top", "right"]].set_visible(False)

    for ax in list(saturation_axes.flat)[len(plot_countries):]:
        ax.set_visible(False)

    legend_handles = [
        Line2D([0], [0], color=group_colours[model], linewidth=2, label=model)
        for model in plot_models
    ]
    legend_handles.append(Line2D(
        [0], [0], marker="o", linestyle="none", markerfacecolor="white",
        markeredgecolor="black", label="Saturation point",
    ))
    saturation_fig.legend(
        handles=legend_handles, loc="outside upper center",
        ncol=len(legend_handles), frameon=False,
    )
    saturation_fig.supxlabel("Return period (years)")
    saturation_fig.supylabel("Baseline population affected")
    plt.show()
